# CNN Quickstart with MNIST

Train a small convolutional neural network on handwritten digits.

## Initialize PyTorch

- Import PyTorch, torchvision, and plotting tools
- Seed PyTorch for reproducible training
- Detect device (GPU, Apple Silicon, CPU)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()
device

## Load the Dataset

- Load MNIST images for CNN training
- Convert images into tensors while keeping channel, height, and width
- Use smaller train and test subsets for quick class runs
- Create `DataLoader`s for batched training and evaluation

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

train_subset = Subset(train_data, range(5000))
test_subset = Subset(test_data, range(1000))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=128)

## Define CNN Model

- Build a convolutional classifier with PyTorch `nn.Module`
- Use convolution layers to learn spatial features from images
- Use pooling to reduce image size between feature stages
- Flatten learned feature maps before final classification
- Output `10` digit class scores

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = MNISTCNN().to(device)
model

## Train and Evaluate CNN

- Define cross-entropy loss and Adam optimizer
- Train for several epochs
- Evaluate test loss and accuracy after each epoch
- Store train/test loss and accuracy for plotting

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += loss_fn(logits, labels).item()
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}

for epoch in range(5):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_accuracy = correct / total
    test_loss, test_accuracy = evaluate(model, test_loader)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, train_accuracy={train_accuracy:.3f}, "
        f"test_loss={test_loss:.4f}, test_accuracy={test_accuracy:.3f}"
    )

## Plot Training Curves

- Plot train and test loss across epochs
- Plot train and test accuracy across epochs
- Use the curves to compare learning and generalization

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["test_loss"], label="test loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross entropy")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs, history["train_accuracy"], label="train accuracy")
axes[1].plot(epochs, history["test_accuracy"], label="test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0, 1)
axes[1].set_title("Accuracy")
axes[1].legend()

plt.tight_layout()

## Visualize Predictions

- Run the trained CNN on one test batch
- Convert class scores to predicted digits
- Display sample images with true and predicted labels

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    predictions = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(9, 3))
for ax, image, label, pred in zip(axes.ravel(), images[:12], labels[:12], predictions[:12]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"true={label.item()} pred={pred.item()}")
    ax.axis("off")
plt.tight_layout()